In [6]:
import numpy as np
import pandas as pd
import yaml

pd.set_option("display.max_columns", None)

# ============================================================
# 1) Load settings
# ============================================================
with open("../../Settings.yaml", "r") as file:
    Setting = yaml.safe_load(file)

# Container for yearly energy frames
Data = []

# ============================================================
# 2) Build Water Usage panel (1383–1400) from multiple files
#    Each raw file is in wide format (years as columns) -> melt to long format
# ============================================================

# -----------------------------
# 2.1) Water usage: 1383–1393 (sheet 7)
# -----------------------------
file_path_1 = f"{Setting['Raw_Path']}/Y1383_Y1393_10AndMore_Water.xlsx"
T7 = pd.read_excel(file_path_1, sheet_name=7, skiprows=3)

years_1 = [str(y) for y in range(1383, 1394)]  # 1383..1393
T7.columns = ["Industry_Code", "Industry_Name"] + years_1

T7 = T7.melt(
    id_vars=["Industry_Code", "Industry_Name"],
    value_vars=years_1,
    var_name="Year",
    value_name="Water (M3)"
)

# Drop the aggregate ("Total industry") row
T7 = T7[T7["Industry_Name"] != "کل صنعت"].copy()

# -----------------------------
# 2.2) Water usage: 1394–1399 (sheet 7)
# -----------------------------
file_path_2 = f"{Setting['Raw_Path']}/Y1394_Y1399_10AndMore_Water.xlsx"
T7_1 = pd.read_excel(file_path_2, sheet_name=7, skiprows=3)

years_2 = [str(y) for y in range(1394, 1400)]  # 1394..1399
T7_1.columns = ["Industry_Code", "Industry_Name"] + years_2

T7_1 = T7_1.melt(
    id_vars=["Industry_Code", "Industry_Name"],
    value_vars=years_2,
    var_name="Year",
    value_name="Water (M3)"
)

T7_1 = T7_1[T7_1["Industry_Name"] != "کل صنعت"].copy()

# -----------------------------
# 2.3) Water usage: 1400 (sheet 8)
# -----------------------------
file_path_3 = f"{Setting['Raw_Path']}/Y1400_10AndMore_Water.xlsx"
T8 = pd.read_excel(file_path_3, sheet_name=8, skiprows=3)

years_3 = ["1400"]
T8.columns = ["Industry_Code", "Industry_Name"] + years_3

T8 = T8.melt(
    id_vars=["Industry_Code", "Industry_Name"],
    value_vars=years_3,
    var_name="Year",
    value_name="Water (M3)"
)

T8 = T8[T8["Industry_Name"] != "کل صنعت"].copy()

# -----------------------------
# 2.4) Stack all water frames together
# -----------------------------
Water_Usage = pd.concat([T7, T7_1, T8], axis=0, ignore_index=True)

# Ensure correct dtypes and panel-friendly ordering
Water_Usage["Year"] = Water_Usage["Year"].astype(int)
Water_Usage = Water_Usage.sort_values(["Industry_Code", "Year"]).reset_index(drop=True)

# ============================================================
# 3) Read yearly Energy usage files and build Energy_Usage panel
#    Note: years 1381, 1382, 1401 are missing in raw energy files
# ============================================================
for year in range(Setting["Start_Year"] + 2, Setting["End_Year"]):
    file_name = f"Y{year}_10AndMore_Energy.xlsx"
    file_path = f"{Setting['Raw_Path']}/{file_name}"

    # Read energy consumption (sheet 11); skip a metadata/header block
    T11 = pd.read_excel(file_path, sheet_name=11, skiprows=1)

    # Standardize column names (must match the sheet's structure)
    cols = [
        "Industry_Code", "Industry_Name",
        "Natural_Gas (M3)", "Fueloil (Liter)", "Gasoil (Liter)", "Gasoline (Liter)",
        "Kerosene (Liter)", "LNG (KG)", "Charcoal and Coal (KG)", "Electricity (KWH)"
    ]
    T11.columns = cols

    # Drop first row (often totals or non-data row in these sheets)
    T11 = T11.iloc[1:].reset_index(drop=True)

    # Add the year identifier to make it panel data
    T11["Year"] = year

    # Collect yearly frames
    Data.append(T11)

# Stack all years into one energy dataset
Energy_Usage = pd.concat(Data, ignore_index=True)

# Drop rows with missing industry code (usually totals/blank rows)
Energy_Usage = Energy_Usage[Energy_Usage["Industry_Code"].notna()]

# ============================================================
# 4) Merge Energy and Water datasets (left join keeps all energy rows)
# ============================================================
Dataset = pd.merge(
    Energy_Usage,
    Water_Usage,
    how="left",
    on=["Year", "Industry_Code"]
)

# Clean duplicate industry name column created by merge
Dataset.drop(columns="Industry_Name_y", inplace=True)
Dataset.rename(columns={"Industry_Name_x": "Industry_Name"}, inplace=True)

# Fill missing water values (and any other missing) with 0
# (This is a strong assumption; keep it only if it matches your methodology.)
Dataset = Dataset.fillna(0)

# Move Year to be the first column
Dataset = Dataset[["Year"] + [c for c in Dataset.columns if c != "Year"]]

# ============================================================
# 5) Create missing years (1381, 1382, 1401) and impute by CAGR
# ============================================================

# Energy-related columns that will be imputed for missing years
energy_cols = [
    "Natural_Gas (M3)", "Fueloil (Liter)", "Gasoil (Liter)", "Gasoline (Liter)",
    "Kerosene (Liter)", "LNG (KG)", "Charcoal and Coal (KG)", "Electricity (KWH)",
    "Water (M3)"
]
missing_years = [1381, 1382, 1401]

# -----------------------------
# 5.1) Load industry codes/names for generating the missing-year grid
#      Keep only Industry_Category_Code == 2
# -----------------------------
input_file = f"{Setting['Workshop_Codes']}"
industries = pd.read_excel(input_file, sheet_name="WorkShop_Codes")
industries = industries[industries["Industry_Category_Code"] == 2]

# Align dtype of Industry_Code with Dataset (common source of merge/indexing bugs)
industries["Industry_Code"] = industries["Industry_Code"].astype("float64")

# Create all combinations (Industry × missing years)
add_rows = industries.merge(pd.DataFrame({"Year": missing_years}), how="cross")

# Initialize target columns as NaN (to be filled by CAGR extrapolation)
for c in energy_cols:
    add_rows[c] = np.nan

# -----------------------------
# 5.2) Prepare numeric dtypes for growth calculations
# -----------------------------
Dataset["Year"] = pd.to_numeric(Dataset["Year"], errors="coerce")
for c in energy_cols:
    Dataset[c] = pd.to_numeric(Dataset[c], errors="coerce")

# -----------------------------
# 5.3) CAGR-based extrapolation per (industry, column)
#      - Use first and last strictly positive observations to compute CAGR
#      - Backcast 1381/1382 and forecast 1401
# -----------------------------
for ind_code in add_rows["Industry_Code"].unique():

    # Historical data for this industry
    base = Dataset[Dataset["Industry_Code"] == ind_code].copy()
    base = base.sort_values("Year")

    # Years to be added for this industry
    add_mask = (add_rows["Industry_Code"] == ind_code)
    years_to_add = add_rows.loc[add_mask, "Year"].tolist()

    for col in energy_cols:

        # Keep only non-missing and strictly positive values (needed for CAGR)
        s = base[["Year", col]].dropna(subset=["Year", col]).sort_values("Year")
        s = s[s[col] > 0]

        # Need at least two points to estimate a growth rate
        if len(s) < 2:
            continue

        # Endpoints for CAGR
        y0 = int(s.iloc[0]["Year"]);  v0 = float(s.iloc[0][col])
        y1 = int(s.iloc[-1]["Year"]); v1 = float(s.iloc[-1][col])

        # Guard against invalid cases
        if y1 == y0 or v0 <= 0 or v1 <= 0:
            continue

        # CAGR formula
        cagr = (v1 / v0) ** (1 / (y1 - y0)) - 1

        # Extrapolate each missing year
        vals = []
        for y in years_to_add:
            y = int(y)

            if y < y0:
                # Backcast from first observed point
                vals.append(v0 / ((1 + cagr) ** (y0 - y)))
            elif y > y1:
                # Forecast from last observed point
                vals.append(v1 * ((1 + cagr) ** (y - y1)))
            else:
                # Safe fallback (not expected for your missing_years list)
                vals.append(v0 * ((1 + cagr) ** (y - y0)))

        # Write extrapolated values back to add_rows
        add_rows.loc[add_mask, col] = vals

# -----------------------------
# 5.4) Append imputed rows; enforce one row per (Industry_Code, Year)
# -----------------------------
Dataset = pd.concat([Dataset, add_rows], ignore_index=True)

Dataset = (
    Dataset
    .drop_duplicates(subset=["Industry_Code", "Year"], keep="first")
    .sort_values(["Year", "Industry_Code"])
    .reset_index(drop=True)
)

# Drop category code (came from industries code file; not needed in final dataset)
Dataset.drop(columns="Industry_Category_Code", inplace=True)

# Final missing-value fill (again: assumes remaining NaNs should be 0)
Dataset = Dataset.fillna(0)



# ============================================================
# 6) Convert fuels to BOE and compute total fuel consumption (BOE)
# ============================================================

# Fuel columns to be converted to BOE
fuel_cols = [
    "Natural_Gas (M3)", "Fueloil (Liter)", "Gasoil (Liter)", "Gasoline (Liter)",
    "Kerosene (Liter)", "LNG (KG)", "Charcoal and Coal (KG)","Electricity (KWH)"
]

# BOE conversion factors (boe per unit)
boe_factor = {
    "Natural_Gas (M3)": 0.00609,             # boe per m3
    "Fueloil (Liter)": 0.00000662,           # boe per liter
    "Gasoil (Liter)": 0.00000585,            # boe per liter
    "Gasoline (Liter)": 0.00000540,          # boe per liter
    "Kerosene (Liter)": 0.00000600,          # boe per liter
    "LNG (KG)": 0.00817,                     # boe per kg
    "Charcoal and Coal (KG)": 0.00408,       # boe per kg
    "Electricity (KWH)": 0.000588            # boe per kwh
}

# Compute BOE per fuel and then sum across fuels
for c in fuel_cols:
    Dataset[f"{c} (boe)"] = Dataset[c] * boe_factor[c]

Dataset["Energy (boe)"] = Dataset[[f"{c} (boe)" for c in fuel_cols]].sum(axis=1)

# Drop intermediate BOE columns (keep only Energy and Electricity (boe))
Dataset.drop(
    columns=[f"{c} (boe)" for c in fuel_cols if c != "Electricity (KWH)"],
    inplace=True,
    errors="ignore"
)
Dataset.rename(columns={'Electricity (KWH) (boe)':'Electricity (boe)'},inplace=True)

Dataset['Elec_boe_intensity'] = Dataset['Electricity (boe)'] / Dataset['Energy (boe)']




# ============================================================
# 7) Export to Excel
# ============================================================
output_file_name = "Unadjusted.xlsx"
sheet_name = "Energy_By_Activity"
path = f"{Setting['Output_Path_Unajusted']}/{output_file_name}"

# Write/replace the target sheet (append mode keeps other sheets in the workbook)
with pd.ExcelWriter(path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    Dataset.to_excel(writer, sheet_name=sheet_name, index=False)

c:\Users\hwi\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
c:\Users\hwi\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
c:\Users\hwi\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
c:\Users\hwi\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
c:\Users\hwi\AppData\Local\Progr